## What does each parameter do?

The most direct diagnostic is

$$
\left|\frac{P_{\delta I}}{P_{\rm m}}\right|=|F(k,z)|.
$$

This removes the matter-spectrum shape and displays the IA response itself.

| Parameter | Main visible effect |
|---|---|
| $A_0$ | rescales everything; changing its sign changes $P_{\delta I}$'s sign |
| $\eta$ | sets the baseline redshift power-law slope |
| $\xi$ | changes the redshift slope by $\xi$ across the luminosity transition |
| $z_q$ | moves the luminosity transition in redshift |
| $s$ | makes the luminosity transition broader or sharper |
| $q$ | changes the high-$k$ amplitude |
| $k_t$ | moves the scale transition left or right |
| $n$ | makes the scale transition smoother or sharper |

In [ ]:
z_test = numpy.asarray([1.2])

experiments = [
    ("A0", [0.5, 1.0, 2.0]),
    ("eta", [-2.0, 0.0, 2.0]),
    ("xi", [-2.0, 0.0, 2.0]),
    ("z_q", [0.4, 1.0, 1.8]),
    ("luminosity_sharpness", [1.0, 3.0, 8.0]),
    ("q", [-0.5, 0.0, 1.5]),
    ("k_transition", [0.05, 0.2, 0.8]),
    ("scale_sharpness", [1.0, 2.0, 4.0]),
]

baseline = {
    "A0": 1.0,
    "eta": 0.0,
    "xi": 1.0,
    "z_q": 1.0,
    "luminosity_sharpness": 3.0,
    "q": 1.0,
    "k_transition": 0.2,
    "scale_sharpness": 2.0,
}

fig, axes = pyplot.subplots(2, 4, figsize=(19, 9))

for ax, (parameter_name, values) in zip(axes.flat, experiments):
    for value in values:
        parameters = baseline.copy()
        parameters[parameter_name] = value
        response, _ = ia_response(
            cosmology,
            z_test,
            k,
            **parameters,
        )
        ax.plot(
            k,
            numpy.abs(response[0]),
            label=f"{parameter_name}={value}",
        )

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$k$ [Mpc$^{-1}$]")
    ax.set_ylabel(r"$|P_{\delta I}/P_{\rm m}|$")
    ax.set_title(f"Vary {parameter_name}")
    ax.legend(fontsize=8)

fig.suptitle(
    r"One parameter at a time, evaluated at $z=1.2$",
    fontsize=15,
)
fig.tight_layout()
pyplot.show()

## 12. Sample parameters and create an ML-ready dataset

Each sampled parameter set creates two surfaces:

$$
P_{\delta I}(z,k)
\quad\text{and}\quad
P_{II}(z,k).
$$

We store them as two channels:

$$
X.\mathrm{shape}
=
(N_{\rm models},\,2,\,N_z,\,N_k).
$$

This example uses independent uniform sampling for clarity. A larger project should use
Latin-hypercube or Sobol sampling for more uniform coverage.

The ranges below are broad teaching ranges, not observational priors.

In [ ]:
rng = numpy.random.default_rng(2026)
number_of_models = 300

parameter_names = [
    "A0",
    "eta",
    "xi",
    "z_q",
    "luminosity_sharpness",
    "q",
    "k_transition",
    "scale_sharpness",
]

theta = numpy.column_stack([
    rng.uniform(0.2, 3.0, number_of_models),          # A0
    rng.uniform(-2.0, 2.0, number_of_models),         # eta
    rng.uniform(-3.0, 3.0, number_of_models),         # xi
    rng.uniform(0.2, 2.0, number_of_models),          # z_q
    rng.uniform(1.0, 8.0, number_of_models),          # luminosity s
    rng.uniform(-0.5, 2.0, number_of_models),         # q
    10.0**rng.uniform(-2.0, -0.2, number_of_models),  # k_t
    rng.uniform(1.0, 4.0, number_of_models),          # scale n
])

k_training = numpy.logspace(-3, 0.5, 96)
z_training = numpy.asarray([0.2, 0.5, 0.8, 1.2, 1.6, 2.0])

P_m_training = numpy.vstack([
    pyccl.nonlin_matter_power(
        cosmology,
        k_training,
        1.0 / (1.0 + z_value),
    )
    for z_value in z_training
])

P_deltaI_training = numpy.empty(
    (number_of_models, len(z_training), len(k_training))
)
P_II_training = numpy.empty_like(P_deltaI_training)

for model_index, values in enumerate(theta):
    parameters = dict(zip(parameter_names, values))
    response, _ = ia_response(
        cosmology,
        z_training,
        k_training,
        **parameters,
    )

    P_deltaI_training[model_index] = response * P_m_training
    P_II_training[model_index] = response**2 * P_m_training

X = numpy.stack(
    [P_deltaI_training, P_II_training],
    axis=1,
)

print("theta shape:", theta.shape)
print("X shape:    ", X.shape)
print("channels:    [P_deltaI, P_II]")

### 12.1 Inspect a few randomly generated models

These plots are a basic quality check. A training dataset should be inspected before it
is given to PCA or a neural network.

In [ ]:
selected_models = [0, 1, 2, 3, 4]
z_index = 2

fig, axes = pyplot.subplots(1, 2, figsize=(13, 5))

for model_index in selected_models:
    axes[0].plot(
        k_training,
        -P_deltaI_training[model_index, z_index],
        label=f"model {model_index}",
    )
    axes[1].plot(
        k_training,
        P_II_training[model_index, z_index],
        label=f"model {model_index}",
    )

axes[0].set_title(
    rf"$-P_{{\delta I}}$ at $z={z_training[z_index]}$"
)
axes[1].set_title(
    rf"$P_{{II}}$ at $z={z_training[z_index]}$"
)

for ax in axes:
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(r"$k$ [Mpc$^{-1}$]")
    ax.set_ylabel(r"power spectrum [Mpc$^3$]")
    ax.legend(fontsize=8)

fig.tight_layout()
pyplot.show()

### 12.2 Save the raw dataset

Save raw signed spectra before applying logarithms or normalization. Preprocessing belongs
in the next lecture and should be reproducible from the saved physical quantities.

In [ ]:
if Path.cwd().name == "Code":
    output_directory = Path.cwd().parent / "Data"
else:
    output_directory = Path.cwd() / "Data"

output_directory.mkdir(parents=True, exist_ok=True)
output_file = output_directory / "nla_ia_training_spectra.npz"

numpy.savez_compressed(
    output_file,
    k=k_training,
    z=z_training,
    theta=theta,
    parameter_names=numpy.asarray(parameter_names),
    P_m=P_m_training,
    P_deltaI=P_deltaI_training,
    P_II=P_II_training,
    X=X,
)

print("Saved:", output_file.resolve())

## Final takeaway

A sampled power spectrum is a long numerical vector. By varying physically meaningful
parameters, we have produced a family of related vectors. That family is the input to
the compression models introduced in the next phase of the project.